<a href="https://colab.research.google.com/github/vidya100804/AI-ML/blob/main/UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# INSTALL (RUN ONCE)
# ============================================================
!pip install xgboost catboost tensorflow --quiet

# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
from catboost import CatBoostClassifier

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_parquet("/content/telugu_news_dataset.parquet")

df.fillna('', inplace=True)
df['combined'] = df['title'] + " " + df['text']

# ============================================================
# CLEAN TEXT
# ============================================================
def clean_text(text):
    text = re.sub(r'[^ఀ-౿\w\s]', ' ', str(text))
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

df['combined'] = df['combined'].apply(clean_text)

# ============================================================
# LABELS (BINARY)
# ============================================================
top_categories = df['category'].value_counts().nlargest(2).index.tolist()
df = df[df['category'].isin(top_categories)]

label_map = {top_categories[0]: 0, top_categories[1]: 1}
df['label'] = df['category'].map(label_map)

X = df['combined']
y = df['label']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

# ============================================================
# TF-IDF
# ============================================================
tfidf = TfidfVectorizer(max_features=20000)
X_train_tfidf = tfidf.fit_transform(X_train)

# ============================================================
# ML MODELS (6)
# ============================================================
models = {
    "XGBoost": xgb.XGBClassifier(n_estimators=200),
    "Random Forest": RandomForestClassifier(n_estimators=200),
    "Decision Tree": DecisionTreeClassifier(max_depth=10),
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "CatBoost": CatBoostClassifier(verbose=0)
}

trained_models = {}

print("\n===== TRAINING ML MODELS =====")
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_tfidf, y_train)
    trained_models[name] = model

# ============================================================
# LSTM (FAST VERSION)
# ============================================================
MAX_WORDS = 40000
MAX_LEN = 300

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN)

lstm_model = Sequential([
    Embedding(MAX_WORDS, 128),  # reduced size (faster)
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.4),
    Bidirectional(LSTM(32)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

print("\n===== TRAINING LSTM (FAST) =====")
lstm_model.fit(
    X_train_pad,
    y_train,
    epochs=2,
    batch_size=128
)

# ============================================================
# SAVE MODELS
# ============================================================
pickle.dump(trained_models, open("ml_models.pkl", "wb"))
pickle.dump(tfidf, open("tfidf.pkl", "wb"))
pickle.dump(tokenizer, open("tokenizer.pkl", "wb"))

# IMPORTANT: save in new format
lstm_model.save("lstm_model.keras")

print("\n✅ ALL MODELS SAVED SUCCESSFULLY")
print("👉 Now use Phase 2 (UI code) only")


===== TRAINING ML MODELS =====
Training XGBoost...
Training Random Forest...
Training Decision Tree...
Training Naive Bayes...
Training Logistic Regression...
Training CatBoost...

===== TRAINING LSTM (FAST) =====
Epoch 1/2
65/65 ━━━━━━━━━━━━━━━━━━━━ 134s 2s/step - accuracy: 0.8767 - loss: 0.2631
Epoch 2/2
65/65 ━━━━━━━━━━━━━━━━━━━━ 124s 2s/step - accuracy: 0.9903 - loss: 0.0354

✅ ALL MODELS SAVED SUCCESSFULLY
👉 Now use Phase 2 (UI code) only


In [7]:
# ============================================================
# UI CODE (RUN THIS ONLY AFTER TRAINING)
# ============================================================
import pickle
import numpy as np
import re
from newspaper import Article
from youtube_transcript_api import YouTubeTranscriptApi
from urllib.parse import urlparse

import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

import gradio as gr

# =========================
# LOAD MODELS
# =========================
ml_models = pickle.load(open("ml_models.pkl", "rb"))
tfidf = pickle.load(open("tfidf.pkl", "rb"))
tokenizer = pickle.load(open("tokenizer.pkl", "rb"))
lstm_model = tf.keras.models.load_model("lstm_model.keras")

# =========================
# CLEAN TEXT
# =========================
def clean_text(text):
    text = re.sub(r'[^ఀ-౿\w\s]', ' ', str(text))
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

# =========================
# TRUSTED DOMAIN CHECK
# =========================
def is_trusted_domain(url):
    trusted_domains = [
        "eenadu.net",
        "sakshi.com",
        "andhrajyothy.com",
        "tv9telugu.com",
        "ntvtelugu.com",
        "10tv.in",
        "hmtvlive.com"
    ]
    domain = urlparse(url).netloc.lower()
    return any(td in domain for td in trusted_domains)

# =========================
# PREDICTION FUNCTION
# =========================
def predict_news(mode, input_data):

    text = ""
    trusted = False

    try:
        # =========================
        # INPUT HANDLING
        # =========================
        if mode == "text":
            text = input_data

        elif mode == "url":
            trusted = is_trusted_domain(input_data)

            article = Article(input_data)
            article.download()
            article.parse()
            text = article.title + " " + article.text

        elif mode == "youtube":
            vid = input_data.split("v=")[-1]
            transcript = YouTubeTranscriptApi.get_transcript(vid)
            text = " ".join([t['text'] for t in transcript])

        if not text.strip():
            return "Prediction: Unable to analyze"

        # =========================
        # CLEAN
        # =========================
        cleaned = clean_text(text)

        # =========================
        # TF-IDF
        # =========================
        X = tfidf.transform([cleaned])

        votes = []

        # =========================
        # ML MODELS
        # =========================
        for model in ml_models.values():
            pred = model.predict(X)[0]
            votes.append(pred)

        # =========================
        # LSTM
        # =========================
        seq = tokenizer.texts_to_sequences([cleaned])
        padded = pad_sequences(seq, maxlen=300)

        lstm_prob = lstm_model.predict(padded)[0][0]
        lstm_pred = 1 if lstm_prob > 0.5 else 0

        votes.append(lstm_pred)

        # =========================
        # FINAL DECISION
        # =========================
        final = max(set(votes), key=votes.count)

        # =========================
        # TRUST LOGIC
        # =========================
        if trusted:
            result = "REAL NEWS"
        elif final == 0:
            result = "REAL NEWS"
        else:
            result = "FAKE NEWS"

        return f"Prediction: {result}"

    except Exception as e:
        return f"Error: {str(e)}"

# =========================
# UI FUNCTION (UNCHANGED)
# =========================
def predict_from_ui(text, url, youtube_url):
    if text:
        return predict_news("text", text)
    elif url:
        return predict_news("url", url)
    elif youtube_url:
        return predict_news("youtube", youtube_url)
    else:
        return "Please enter Text or URL or YouTube URL"

# =========================
# UI (ONLY RESULT BOX SIZE CHANGED)
# =========================
interface = gr.Interface(
    fn=predict_from_ui,
    inputs=[
        gr.Textbox(label="Enter Telugu News Text"),
        gr.Textbox(label="Enter News URL"),
        gr.Textbox(label="Enter YouTube URL")
    ],
    outputs=gr.Textbox(
        label="Result",
        lines=6
    ),
    title="Telugu News Verifier",
    description="Determine if a piece of news is REAL or FAKE using domain verification and pattern analysis."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://75bb2f63f76a755036.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
